In [ ]:
#use "../lib/nb.ml";;
run_notebook "Types.ipynb"

In [ ]:
#show_type regexp;;
#show_type dfa

# Converting a <span style="font-variant:small-caps;">Dfa</span> into a Regular Expression

The function `regexp_sum` takes a list $S = [ r_1, \cdots, r_n ]$ of regular expressions
as its argument.  It returns the regular expression 
$$ r_1 + \cdots + r_n. $$
The regular expression is represented as a value of the type `regexp`, which is defined in the notebook
`Types.ipynb`.  In Python, the argument can be a set or a list.  In OCaml, it is a list, and sets are
converted into lists with `FSet.to_list`.

**Function `regexp_sum s`**
- *Input:* `s` is a list of regular expressions.
- *Output:* The regular expression $r_1 + \cdots + r_n$, where $r_1, \cdots, r_n$ are the elements of `s`.  If `s` is empty, the result is `Empty`, which denotes $\emptyset$.

In [ ]:
let rec regexp_sum (s : regexp list) : regexp =
  match s with
  | [] -> Empty
  | [ r ] -> r
  | r :: rs -> Sum (r, regexp_sum rs)

The function `rpq` assumes there is some <span style="font-variant:small-caps;">Dfa</span>
$$ F = \langle \texttt{States}, \Sigma, \delta, \texttt{q0}, \texttt{Accepting} \rangle $$
given and takes five arguments:
- `p1` and `p2` are states of the <span style="font-variant:small-caps;">Dfa</span> $F$,
- $\Sigma$ is the alphabet of the <span style="font-variant:small-caps;">Dfa</span>,
- $\delta$ is the transition function of the <span style="font-variant:small-caps;">Dfa</span> $F$, and
- `Allowed` is a subset of the set `States`.  On recursive calls, `Allowed` is a list of states.

The function `rpq` computes a regular expression that describes those strings that take the 
<span style="font-variant:small-caps;">Dfa</span> $F$ from the state `p1` to state `p2`.
When $F$ switches states from `p1` to `p2` only states in the set `Allowed` may be visited in-between the states `p1` and `p2`.

The function is defined by recursion on the set `Allowed`.  There are two cases
- $\texttt{Allowed} = \{\}$.  
  Define `AllChars`as the set of all characters that when read by $F$ in the state `p1` cause $F$ to enter the state `p2`:
  $$ \texttt{AllChars} = \{ c \in \Sigma \mid \delta(p_1, c) = p_2 \} $$
  Then we need a further case distinction:
  - $p_1 = p_2$: In this case we have:
    $$ \texttt{rpq}(p_1, p_2, \{\}) := \sum\limits_{c\in\texttt{AllChars}} c \quad + \varepsilon$$
    If $\texttt{AllChars} = \{\}$ the sum $\sum\limits_{c\in\texttt{AllChars}} c$ is to be interpreted as the
    regular expression $\emptyset$ that denotes the empty language. 
    
    Otherwise, if $\texttt{AllChars} = \{c_1,\cdots,c_n\}$ we have
    $\sum\limits_{c\in\texttt{AllChars}} c \quad = c_1 + \cdots + c_n$.
  - $p_1 \not= p_2$: In this case we have:
    $$ \texttt{rpq}(p_1, p_2, \{\}) := \sum\limits_{c\in\texttt{AllChars}} c \quad$$
- $\texttt{Allowed} = \{ q \} \cup \texttt{RestAllowed}$.  In this case we recursively define the following variables:
  1. $\texttt{rp1p2} := \texttt{rpq}(p_1, p_2, \Sigma, \delta, \texttt{RestAllowed})$,
  2. $\texttt{rp1q } := \texttt{rpq}(p_1, q, \Sigma, \delta, \texttt{RestAllowed})$,
  3. $\texttt{rqq }\texttt{ } := \texttt{rpq}(q, q, \Sigma, \delta, \texttt{RestAllowed})$,
  4. $\texttt{rqp2 } := \texttt{rpq}(q, p_2, \Sigma, \delta, \texttt{RestAllowed})$.

  Then we can define:
  $$ \texttt{rpq}(p_1, p_2, \texttt{Allowed}) := \texttt{rp1p2} + \texttt{rp1q} \cdot \texttt{rqq}^* \cdot \texttt{rqp2} $$
  This formula can be understood as follows:  If a string $w$ is read in state $p_1$ and reading this string takes the 
  <span style="font-variant:small-caps;">Dfa</span> $F$ from the state $p_1$ to the state $p_2$ while only visiting states from the set 
  `Allowed` in-between, then there are two cases:
  - Reading $w$ does not visit the state $q$ in-between.  Hence the string $w$ can be described by the regular expression
    `rp1p2`.
  - The string $w$ can be written as $w = t u_1 \cdots u_n v$ where:
    - reading $t$ in the state $p_1$ takes the <span style="font-variant:small-caps;">Dfa</span> $F$ into the state $q$,
    - for all $i \in \{1,\cdots,n\}$ reading $u_i$ in the state $q$ takes the <span style="font-variant:small-caps;">Dfa</span> $F$ from $q$ to $q$, and
    - reading $v$ in the state $q$ takes the <span style="font-variant:small-caps;">Dfa</span> $F$ into the state $p_2$.

**Function `rpq p1 p2 sigma delta allowed`**
- *Input:* `p1` and `p2` are states of a <span style="font-variant:small-caps;">Dfa</span> $F$, `sigma` is its alphabet, `delta` is its transition function, and `allowed` is a list of states of $F$.
- *Output:* A regular expression that describes the strings that take $F$ from `p1` to `p2` while visiting only states from `allowed` in-between.

In OCaml, the set `Allowed` is passed as a list, so the Python statement `q, *RestAllowed = Allowed` becomes the
pattern `q :: restAllowed`.

In [ ]:
let rec rpq (p1 : 'q) (p2 : 'q) (sigma : char FSet.t) (delta : 'q trans_rel_det) (allowed : 'q list) : regexp =
  match allowed with
  | [] ->
      let allChars = FSet.filter (fun c -> Dict.find_opt (p1, c) delta = Some p2) sigma in
      let r = regexp_sum (List.map (fun c -> Char c) (FSet.to_list allChars)) in
      if p1 = p2 then Sum (Epsilon, r) else r
  | q :: restAllowed ->
      let rp1p2 = rpq p1 p2 sigma delta restAllowed in
      let rp1q  = rpq p1 q  sigma delta restAllowed in
      let rqq   = rpq q  q  sigma delta restAllowed in
      let rqp2  = rpq q  p2 sigma delta restAllowed in
      Sum (rp1p2, Product (Product (rp1q, Star rqq), rqp2))

The function `dfa_2_regexp` takes a <span style="font-variant:small-caps;">Dfa</span> $F$ and computes a regular expression $r$ that describes the same language as $F$, i.e. we have
$$ L(F) = L(r). $$

**Function `dfa_2_regexp f`**
- *Input:* `f` is a <span style="font-variant:small-caps;">Dfa</span>.
- *Output:* A regular expression $r$ such that $L(r) = L(F)$.

In [ ]:
let dfa_2_regexp (f : 'q dfa) : regexp =
  let ({ states; sigma; delta; start = q0; accepting } : 'q dfa) = f in
  regexp_sum (FSet.to_list (FSet.map (fun p -> rpq q0 p sigma delta (FSet.to_list states)) accepting))

The notebook `06-Test-DFA-2-RegExp.ipynb` provides a test for the function `dfa_2_regexp`.